# W05 · 02 평형 판정 · 블록 평균 · 자기상관 — W03 온도 시계열
자료: `data/T_series_week03.csv` (W03 LJ 유체 64입자 NVE · 1,001점 · Δt 1 fs). 목표 3
1. t_eq 탐색: 폐기 구간을 늘리며 직후 100스텝 평균과 나머지 평균의 차가 2 SE 안에 드는 첫 지점 (슬라이드 16 기준 ①②)
2. 블록 평균 SE(b) 곡선 → 평탄값 = 참 SE (슬라이드 21)
3. 자기상관 C(τ) → τ_c → N_eff = T/2τ_c (슬라이드 20) → `out/block_stats.json`

In [ ]:
# 기본 = 로컬 mdai · Claude Code 실행(아래 if 는 건너뜀) · Colab 은 선택: 저장소 clone 후 week05 로 이동
import os, sys, json, math
if "google.colab" in sys.modules and not os.path.exists("scripts/md_stats.py"):
    !git clone https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git repo && %cd repo/week05
import numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "scripts")
from md_stats import *   # LJFluid thermostats equilibration block_se autocorr OUT

In [ ]:
res = equilibration(); print(json.dumps(res, indent=1, ensure_ascii=False))
# expected_output/block_stats.json: t_eq 60 · τ_c ≈ 27 스텝 · N_eff ≈ 17 · 블록 SE 평탄값 ≈ 2.3 K

In [ ]:
d = np.loadtxt("data/T_series_week03.csv", delimiter=",", skiprows=1); T = d[:, 5]; teq = res["t_eq_step"]; x = T[teq:]
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].plot(T, lw=1); ax[0].axvspan(0, teq, color="0.85"); ax[0].axhline(x.mean(), color="r", ls="--"); ax[0].set_title("T(t) · shaded = discarded"); ax[0].set_xlabel("step")
bs = [1, 2, 3, 5, 7, 10, 15, 20, 30, 50, 70, 100, 150, 200]; ax[1].semilogx(bs, [block_se(x, b) for b in bs], "o-"); ax[1].set_title("block SE vs block size"); ax[1].set_xlabel("b"); ax[1].set_ylabel("SE (K)")
ac = np.loadtxt(OUT / "autocorr.csv", delimiter=",", skiprows=1); ax[2].plot(ac[:, 0], ac[:, 1]); ax[2].axhline(0, color="gray", lw=0.8); ax[2].set_title(f"autocorrelation · tau_c {res['tau_c_steps']} steps"); ax[2].set_xlabel("lag (step)")
fig.tight_layout(); fig.savefig(OUT / "equilibration_stats.png", dpi=120); plt.show()

## 읽기
- 순진한 SE σ/√n = 0.30 K 는 상관된 표본을 독립으로 센 값 · 블록 SE 평탄값 ≈ 2.3 K 가 참값 (8배 차이)
- N_eff = T/2τ_c ≈ 17 — 1,001점 궤적이 독립 표본 17개 · 온도 평균의 오차 ±2 K
- 점도의 응력 자기상관은 τ_c 가 훨씬 길다 → W09 에서 별도 취급
- 보고값 = 평균 ± SE(블록) · 시드 3개 이상 평균 (슬라이드 18)